In [1]:
import pandas as pd
import numpy as np
import re
import statsmodels.formula.api as smf
import os

# 제출본 주기: 원본의 os.chdir(개인 PC 경로)를 제거하고 데이터 경로만 상대경로로 수정 (분석 로직 동일)

# 1. 데이터 로드 및 폭력범죄 필터링
df = pd.read_csv("../Dataset/tagged_file_0.csv", encoding="cp949")
violent_tags = ['violence', 'injury', 'robbery', '폭행', '상해', '협박', '폭력']
df['is_violent'] = df['tag_crime_group_manual'].apply(
    lambda x: any(k in str(x) for k in violent_tags)
)
df_viol = df[df['is_violent'] & df['guilty'].str.contains('Guilty', na=False, case=False)].copy()

# 2. 폭력범죄 양형기준상 분류군 태깅
def tag_violent_subgroup(text):
    text = str(text)
    if '특수상해' in text: return '특수상해'
    if '존속상해' in text: return '존속상해'
    if '상해' in text: return '일반상해'
    if '특수폭행' in text: return '특수폭행'
    if '폭행' in text: return '일반폭행'
    if '특수협박' in text: return '특수협박'
    if '협박' in text: return '일반협박'
    return '기타폭력'
df_viol['sentencing_group'] = df_viol['tag_crime_name_manual'].apply(tag_violent_subgroup)

# 3. 형량 추출 (개월 수)
def extract_sentence(text):
    match = re.search(r'징역\s*(\d+)년(?:\s*(\d+)월)?', str(text))
    if match:
        return int(match.group(1)) * 12 + (int(match.group(2)) if match.group(2) else 0)
    return np.nan
df_viol['sentence_months'] = df_viol['full_text'].apply(extract_sentence)
df_viol['is_suspended'] = df_viol['full_text'].apply(lambda x: '집행유예' in str(x))

# 4. 상해 피해도 (전치 OO주) 추출
def extract_injury_weeks(text):
    match = re.search(r'전치\s*약\s*(\d+)\s*주|전치\s*(\d+)\s*주|치료를\s*요하는\s*약\s*(\d+)\s*주|(\d+)\s*주간의\s*치료', str(text))
    if match:
        for g in match.groups():
            if g is not None:
                return int(g)
    return 0
df_viol['injury_weeks'] = df_viol['full_text'].apply(extract_injury_weeks)

# 5. 폭력범죄 맞춤 감경/가중요소 태깅
df_viol['factor_reflection'] = df_viol['full_text'].apply(lambda x: 1 if '반성' in str(x) else 0)
df_viol['factor_agreement'] = df_viol['full_text'].apply(lambda x: 1 if '합의' in str(x) else 0)
df_viol['factor_non_punishment'] = df_viol['full_text'].apply(lambda x: 1 if '처벌불원' in str(x) or '처벌을 원하지' in str(x) else 0)
df_viol['factor_repeat_offense'] = df_viol['full_text'].apply(lambda x: 1 if '누범' in str(x) or '동종' in str(x) else 0)

# 6 & 7. Outcome 변수 (선고형 < 중앙값 또는 집행유예)
mean_sentence = df_viol['sentence_months'].median()
df_viol['is_mitigated'] = (df_viol['sentence_months'] < mean_sentence) | df_viol['is_suspended']
df_viol['outcome'] = df_viol['is_mitigated'].astype(int)

# 8. 양형 결과 회귀분석 (처벌불원, 누범 요소 등 포함)
model1 = smf.logit("outcome ~ factor_reflection + factor_agreement + factor_non_punishment + factor_repeat_offense", data=df_viol).fit()
print(model1.summary())

# 9. 합의 여부 회귀분석 (상해 주수와 반성 여부의 영향)
model2 = smf.logit("factor_agreement ~ factor_reflection + injury_weeks", data=df_viol).fit()
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.532932
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                  340
Model:                          Logit   Df Residuals:                      335
Method:                           MLE   Df Model:                            4
Date:                Sat, 25 Jul 2026   Pseudo R-squ.:                  0.1240
Time:                        13:28:15   Log-Likelihood:                -181.20
converged:                       True   LL-Null:                       -206.84
Covariance Type:            nonrobust   LLR p-value:                 1.946e-10
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -1.6403      0.195     -8.398      0.000      -2.023      -1.